In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv("../1.9 Bike Sharing/data/day.csv")
print(df.shape)
df.head()


(731, 16)


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


Query 1 — Verify Total Observations

The SQLite database contains 731 observations, matching the original Pandas DataFrame. This confirms that all rows from the CSV were successfully imported into the database.

In [3]:
conn = sqlite3.connect("bike_sharing.db")

In [4]:
df.to_sql(name="bike_days", con=conn, if_exists="replace", index=False)

731

In [5]:
pd.read_sql_query("SELECT COUNT(*) FROM bike_days", conn)

,COUNT(*)
0,731


Query 1 — Verify Total Observations

The SQLite database contains 731 observations, matching the original Pandas DataFrame. This confirms that all rows from the CSV were successfully imported into the database.

In [6]:
pd.read_sql_query("SELECT dteday, cnt FROM bike_days WHERE workingday = 1 ORDER BY cnt DESC LIMIT 5", conn)

,dteday,cnt
0,2012-03-23,8362
1,2012-07-25,8173
2,2012-09-21,8167
3,2012-10-05,8156
4,2012-09-14,8009


Query 2 — Highest-Demand Working Days

The working day with the highest number of bicycle rentals was March 23, 2012, with 8,362 rentals. All five highest-demand working days occurred in 2012. However, this alone does not prove that overall demand was higher in 2012.

In [7]:
pd.read_sql_query("SELECT yr, AVG(cnt), COUNT(*) FROM bike_days GROUP BY yr ORDER BY yr ASC", conn)

,yr,AVG(cnt),COUNT(*)
0,0,3405.761644,365
1,1,5599.934426,366


Query 3 — Average Daily Rentals by Year

In 2011, average daily bicycle rentals were approximately 3,406, compared with 5,600 in 2012, representing an increase of approximately 64.4%. This indicates that observed average daily rental demand was considerably higher in 2012 than in 2011.

In [8]:
monthly_results = pd.read_sql_query("SELECT yr, mnth, AVG(CNT), COUNT(*) FROM bike_days GROUP BY yr, mnth ORDER BY yr, mnth ASC", conn)

In [9]:
monthly_results.tail(12)

,yr,mnth,AVG(CNT),COUNT(*)
12,1,1,3120.774194,31
13,1,2,3556.448276,29
14,1,3,5318.548387,31
15,1,4,5807.466667,30
16,1,5,6318.225806,31
17,1,6,6761.000000,30
18,1,7,6567.967742,31
19,1,8,6919.451613,31
20,1,9,7285.766667,30
21,1,10,6414.225806,31


Query 4 — Monthly Rental Demand by Year

Every month in 2012 had higher average daily rentals than the corresponding month in 2011. September 2012 had the highest monthly average, with approximately 7,286 rentals per day. However, rental demand did not increase continuously from month to month, suggesting that seasonal patterns may influence demand.

In [10]:
pd.read_sql_query("SELECT workingday, AVG(cnt) AS average_rentals, COUNT(*) AS total_days FROM bike_days GROUP BY workingday ORDER BY workingday ASC", conn)

,workingday,average_rentals,total_days
0,0,4330.168831,231
1,1,4584.820000,500


Query 5 — Working Days vs. Non-Working Days

Working days averaged approximately 4,585 rentals per day, compared with 4,330 on non-working days. Working days therefore had approximately 5.9% higher average demand. However, this relationship does not establish causation because other factors, including weather and seasonality, may influence bicycle rental demand.

In [11]:
pd.read_sql_query("SELECT weathersit, AVG(cnt) AS average_rentals, COUNT(*) AS total_days FROM bike_days GROUP BY weathersit ORDER BY AVG(cnt) DESC", conn)

,weathersit,average_rentals,total_days
0,1,4876.786177,463
1,2,4035.862348,247
2,3,1803.285714,21


Query 6 — Weather Conditions and Rental Demand

Clear or partly cloudy conditions had the highest average daily rentals (4,877), followed by misty or cloudy conditions (4,036). Rainy or snowy conditions had the lowest average (1,803).

However, the rainy/snowy category contained only 21 observations, compared with 463 clear-weather observations. This smaller sample increases uncertainty about the estimated average. Furthermore, the association does not establish that weather alone caused the difference in demand.

In [12]:
pd.read_sql_query("SELECT season, AVG(cnt) AS average_rentals, COUNT(*) AS total_days FROM bike_days GROUP BY season ORDER BY AVG(cnt) DESC", conn)

,season,average_rentals,total_days
0,3,5644.303191,188
1,2,4992.331522,184
2,4,4728.162921,178
3,1,2604.132597,181


Query 7 — Seasonal Rental Demand

Fall had the highest average daily bicycle rentals (5,644), followed by summer (4,992), winter (4,728), and spring (2,604).

These findings suggest that rental demand varies substantially across seasons. However, we cannot conclude that temperature alone explains these differences because precipitation, daylight, holidays, and other factors may also influence demand.

In [13]:
pd.read_sql_query("SELECT COUNT(*) AS mismatch_days FROM bike_days WHERE casual + registered != cnt", conn)

,mismatch_days
0,0


Query 8 — Detect Target Leakage

The SQL validation returned zero mismatched observations, confirming that casual + registered = cnt for all 731 days.

Since cnt represents total rentals, including casual and registered as input features when predicting future rental demand would introduce target leakage. These values would not be available when predicting tomorrow's rentals, potentially producing unrealistically high model performance during evaluation.

Overall Conclusions

The SQL exploration identified several important patterns:

Average daily bicycle rentals increased substantially from 2011 to 2012.

Rental demand differed across months, seasons, working-day categories, and weather conditions.

Observed associations do not necessarily establish causal relationships.

Unequal sample sizes should be considered when evaluating the reliability of group averages.

Features must be evaluated for target leakage and availability at prediction time before being used in an ML model.

Skills practiced: SELECT, WHERE, ORDER BY, LIMIT, COUNT(), AVG(), GROUP BY, column aliases, SQLite data ingestion, data validation, and exploratory analysis.